# RETECO · Transfer of insufficient-context detectors

Do insufficient-context detectors trained on synthetic unanswerability **transfer to
natural unanswerability**, where the answer is missing because retrieval failed (RETECO 2c)?

Detectors, all read at the `<|status_start|>` position after OCC's reasoning trace
(names in the output tables in parentheses):
- **verdict** (`OCC verdict`): what OCC itself wrote (ANSWERABLE / UNANSWERABLE)
- **p_status** (`p_status`): probability of ANSWERABLE at that position, no training
- **trace** (`trace`): logistic regression on the trace length and its mean log-probability
- **probe** (`probe`): logistic regression on the hidden state of a single layer

Training (`train` column): on synthetic data (`synthetic`: SQuAD 2.0 + MuSiQue, train), which
measures transfer; and on RETECO train (`RETECO train`), the in-domain ceiling; `—` = no
training. Evaluation: RETECO dev 2c with the qrels label: 1 if the top-5 contains all the
gold documents that fit, 0 if it contains no gold document.
The layer is selected on a held-out part of the training data; dev is not touched.
The last cell evaluates the same trained detectors under the **content** label (`content`)
from `suff.ipynb` (yes = 1, no = 0) and the **clean** label (`clean`: yes = 1; no = 0 only
if the answer is also absent from the dialogue history), if
`reteco/suff/phi-4_2c-<stage>_dev.jsonl` exists on Drive.

**How to run.** Runtime **A100** or **L4** → **Run all**. If cell 2 asks for a restart,
restart the session and **Run all** again. OCC responses on RETECO are taken from
`reteco/gen` (generate.ipynb); missing ones are generated.
Dev features are stored in one file per first stage: adding a stage to DEV_STAGES does not
recompute the existing ones.

**Detectors only, no GPU.** If all `reteco/transfer/feat_*.npz` already exist → set
`EVAL_ONLY = True` in cell 1, runtime **CPU** → **Run all**: the model is not loaded (only
its tokenizer) and nothing is generated; takes minutes. Stages without features are skipped.

**Cost.** Generation of ≈ 9k OCC responses (A100 ≈ 30 min), then one model pass per
example for the hidden states (≈ 15–30 min); probes take minutes.

**Output.** `reteco/transfer/feat_*.npz`: features;
`reteco/analysis/transfer.csv`: AUROC table (qrels label);
`reteco/analysis/transfer_content.csv`: the same detectors under the qrels, content and
clean labels, per retriever (`stage`) and pooled (stage `total`).

**Next.** Copy `reteco/transfer/` and `reteco/analysis/` from Drive to
`data/drive/reteco/transfer/` and `data/drive/reteco/analysis/` of the repo.

**To keep the session from disconnecting when idle**: after "Run all", open the browser console
(Cmd+Option+J → Console), paste the code below and press Enter. Once a minute it clicks the
resources button while the runtime is connected, and it stops by itself once the notebook has finished
and released the GPU (it does not reconnect, so no compute units are spent). The computer must not
go to sleep, and the tab must stay open.
A log of all output with timestamps is written to `logs/` in the project folder on Drive (`DRIVE`).

```js
// Clicker for Colab: keeps the session from disconnecting when idle while the runtime is connected.
// Paste it into the console of the Colab tab (Cmd+Option+J → Console tab → paste → Enter)
// after the notebook has been started. Once a minute it clicks the resources button in the header.
// It stops by itself once the runtime has disconnected (the notebook finished or crashed and
// released the GPU) and does NOT reconnect, so no compute units are spent on an idle machine.
(() => {
  if (window.__retecoKeepAlive) { console.log("clicker is already running"); return; }
  const btn = () => document.querySelector("colab-connect-button")?.shadowRoot?.querySelector("#connect");
  const disconnected = () => {
    const b = btn();
    if (!b) return true;
    // the button also shows the GPU type ("Reconnect T4"), so only the first word is compared;
    // the non-English labels below are the same button texts in the Russian-locale Colab UI
    const text = (b.textContent || "").trim().split(/\s+/)[0].toLowerCase();
    const tip = (b.getAttribute("tooltiptext") || "").toLowerCase();
    return ["connect", "reconnect", "подключиться", "переподключиться"].includes(text)
      || tip.startsWith("connect to a new") || tip.startsWith("подключиться к");
  };
  let seen = false, clicks = 0;
  window.__retecoKeepAlive = setInterval(() => {
    if (!disconnected()) {
      seen = true;
      btn().click();                       // opens/closes the resources panel; harmless
      clicks += 1;
      console.log(`keep-alive ${new Date().toLocaleTimeString()} · ${clicks}`);
    } else if (seen) {
      clearInterval(window.__retecoKeepAlive);
      window.__retecoKeepAlive = null;
      console.log("runtime disconnected: clicker stopped");
    }
  }, 60 * 1000);
  console.log("clicker started: once a minute; stops by itself after the runtime disconnects");
})();
```

In [ ]:
# ── 1. Settings ─────────────────────────────────────────────────────────────
READER = "occ-ai/OCC-RAG-1.7B"
N_TRAIN, N_VAL = 400, 100   # per class in each synthetic set: training / layer selection
DEV_STAGES = ["bm25_query", "bm25_hist", "bm25_fuse3",          # same as FIRST_STAGES in generate
              "hyb_dense_hanhainebula_reason-embed-qwen3-4b-0928_q2d"]
TRAIN_STAGES = ["bm25_hist", "bm25_fuse3"]      # there is no bm25_query run on train
LAYERS = [4, 8, 12, 16, 20, 24, 28]             # Qwen3-1.7B has 28 layers
K_CTX, MAX_NEW, MAXLEN, TRIM_WORDS, CHUNK = 5, 1200, 16384, 300, 256   # as in generate.ipynb
SEED = 0
SUFF_JUDGE = "phi-4"        # content label: reteco/suff/<judge>_2c-<stage>_dev.jsonl (suff.ipynb)
EVAL_ONLY = False           # True: detectors only, from existing transfer/feat_*.npz; CPU runtime, model not loaded
DRIVE = "/content/drive/MyDrive/RETECO 2026/reteco"   # project folder on Drive
NEED_GPU = not EVAL_ONLY

In [ ]:
# ── 1b. Compute-unit safeguard ──────────────────────────────────────────────
# At the end of the notebook the runtime disconnects itself, so an idle GPU does not burn compute units.
# If a cell fails and nothing is run for {IDLE_STOP_MIN} min, it disconnects as well.
# To turn this off, set AUTO_STOP = False in the settings.
NB = "transfer"

import threading, time
from IPython import get_ipython
AUTO_STOP = globals().get("AUTO_STOP", True)
IDLE_STOP_MIN = globals().get("IDLE_STOP_MIN", 15)

def stop_runtime(reason):
    print(f"\n[{time.strftime('%H:%M:%S')}] {reason}: disconnecting the runtime to save compute units", flush=True)
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as e:
        print("could not disconnect:", e)

_WD = {"t": None}
def _pre_cell(*a, **k):
    if _WD["t"]:
        _WD["t"].cancel(); _WD["t"] = None
def _post_cell(result):
    if AUTO_STOP and getattr(result, "error_in_exec", None) is not None:
        _WD["t"] = threading.Timer(IDLE_STOP_MIN * 60, stop_runtime,
                                   args=(f"error and {IDLE_STOP_MIN} min without activity",))
        _WD["t"].daemon = True; _WD["t"].start()
_ip = get_ipython()
if _ip is not None and not getattr(_ip, "_reteco_guard", False):
    _ip.events.register("pre_run_cell", _pre_cell)
    _ip.events.register("post_run_cell", _post_cell)
    _ip._reteco_guard = True
print("safeguard:", "the runtime disconnects itself at the end or", IDLE_STOP_MIN, "min after an error"
      if AUTO_STOP else "off (AUTO_STOP = False)")

In [ ]:
# ── 2. Drive, vLLM, GPU ─────────────────────────────────────────────────────
import os, sys, subprocess
os.environ["HF_HUB_DISABLE_XET"] = "1"
# JAX (pulled in by bm25s) preallocates 75% of GPU memory by default, and vLLM then fails to start
os.environ.setdefault("JAX_PLATFORMS", "cpu")
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
from google.colab import drive
drive.mount("/content/drive")
PIP_PKGS = "pytrec-eval-terrier" if globals().get("EVAL_ONLY") else "vllm openai pytrec-eval-terrier"
!pip -q install {PIP_PKGS} 2>&1 | tail -2
import torch

# vllm installs its own torch, while Colab's torchvision/torchaudio stay built for a different
# CUDA, and import vllm fails. Fix: uninstall torchaudio and reinstall torchvision
# for the new torch's CUDA. A runtime restart is needed afterwards.
def _imports_ok(mod):
    try:
        __import__(mod); return True
    except RuntimeError:
        return False       # installed, but built for a different CUDA
    except ImportError:
        return True        # not installed, so no conflict
fixed = False
if not _imports_ok("torchaudio"):
    subprocess.run([sys.executable, "-m", "pip", "-q", "uninstall", "-y", "torchaudio"], check=True)
    fixed = True
if not _imports_ok("torchvision") and "+" in torch.__version__:
    base, cu = torch.__version__.split("+")
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "--force-reinstall", "--no-deps",
                    f"torchvision==0.{int(base.split('.')[1]) + 15}.*",
                    "--index-url", f"https://download.pytorch.org/whl/{cu}"], check=True)
    fixed = True
if fixed:
    raise RuntimeError("Dependencies fixed. Runtime → Restart session, "
                       "then Run all again.")
if globals().get("NEED_GPU", True):
    assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → A100 or L4"
MEM = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
DTYPE = "bfloat16" if MEM and torch.cuda.get_device_capability(0)[0] >= 8 else "half"   # T4 (sm75): no bf16
BIG = MEM >= 39                                                             # A100 40/80 GB
MID = MEM >= 22                                                             # L4 24 GB and above
print(torch.cuda.get_device_name(0) if MEM else "no GPU", f"{MEM:.0f} GB, {DTYPE}, torch", torch.__version__)

def load_llm(model, maxlen, util=0.88):
    """Uses the real stdout/stderr while the constructor runs: in a Jupyter kernel vLLM fails
    on sys.stdout.fileno() before the weights are even loaded."""
    from vllm import LLM
    out, err = sys.stdout, sys.stderr
    sys.stdout, sys.stderr = sys.__stdout__, sys.__stderr__
    try:
        return LLM(model=model, dtype=DTYPE, max_model_len=maxlen, gpu_memory_utilization=util, seed=0)
    finally:
        sys.stdout, sys.stderr = out, err

def chat(tok, msgs):
    """Prompt built with the model's chat template; Qwen3 thinking is disabled (fast, deterministic)."""
    return tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False, enable_thinking=False)

In [ ]:
# ── 3. RETECO data, track 2 (≈270 MB, about a minute) ───────────────────────
from huggingface_hub import snapshot_download
snapshot_download("DataScience-UIBK/RETECO-SemEval2027", repo_type="dataset",
                  local_dir="/content/reteco",
                  allow_patterns=["track2_recor/*/documents.jsonl",
                                  "track2_recor/*/benchmark_*.json",
                                  "track2_recor/*/qrels_*.txt"])

In [ ]:
# ── 4. Shared code. Mirrors scripts/retrieve.py and score.py in the repository ──
import os, re, json, time, glob
import numpy as np, pytrec_eval
from tqdm.auto import tqdm

DATA = "/content/reteco/track2_recor"
DOMAINS = ["drones", "politics", "law", "medicalsciences", "hardware", "economics",
           "psychology", "biology", "sustainable_living", "robotics", "earth_science"]  # smallest to largest
# official BM25 "turn + history", dev, nDCG@10, for comparison
BM25_DEV = {"biology": .6065, "drones": .2954, "earth_science": .6264, "economics": .4739,
            "hardware": .3378, "law": .3437, "medicalsciences": .1730, "politics": .3219,
            "psychology": .5726, "robotics": .5123, "sustainable_living": .5531}
BM25_MACRO = {"dev": 0.4379, "train": 0.4539}
RUNS, QUERIES = f"{DRIVE}/runs", f"{DRIVE}/queries"

# First run: the BM25 runs and LLM query rewrites sit next to the notebooks as parts
# of the archive reteco_bundle.tar.xz.part*; they are unpacked into DRIVE once.
_parts = sorted(glob.glob(f"{os.path.dirname(DRIVE)}/reteco_bundle.tar.xz.part*"))
if _parts and not os.path.exists(f"{DRIVE}/.bundle_ok"):
    import tarfile
    with open("/content/bundle.tar.xz", "wb") as _out:
        for _p in _parts:
            _out.write(open(_p, "rb").read())
    with tarfile.open("/content/bundle.tar.xz", "r:xz") as _tf:
        try:
            _tf.extractall(os.path.dirname(DRIVE), filter="data")
        except TypeError:
            _tf.extractall(os.path.dirname(DRIVE))
    open(f"{DRIVE}/.bundle_ok", "w").write("ok")
    print(f"project data unpacked from {len(_parts)} parts → {DRIVE}")
    try:                 # flush the unpacked files to Drive now, before the next notebooks start
        from google.colab import drive as _gd
        _gd.flush_and_unmount(); _gd.mount("/content/drive")
    except Exception as _e:
        print("flush Drive:", _e)
assert os.path.isdir(f"{RUNS}/bm25_hist_dev"), (
    f"no project data in {DRIVE}: the notebooks and reteco_bundle.tar.xz.part00…03 must be "
    "in the parent folder of DRIVE (default: MyDrive/RETECO 2026) of the account mounted by drive.mount")
os.makedirs(RUNS, exist_ok=True); os.makedirs(QUERIES, exist_ok=True)

def load_corpus(dom):
    ids, texts = [], []
    for ln in open(f"{DATA}/{dom}/documents.jsonl", encoding="utf-8"):
        if ln.strip():
            d = json.loads(ln); ids.append(d["doc_id"]); texts.append(d["content"])
    return ids, texts

def topics(dom, split, qset):
    """Turns without gold documents are skipped; history is appended as in the organisers' baseline.
    qset: "query", "hist" or a folder name in Drive/reteco/queries (variants from rewrite.py)."""
    given = None
    if qset not in ("query", "hist"):
        with open(f"{QUERIES}/{qset}/{split}.jsonl", encoding="utf-8") as f:
            # key (domain, topic_id): in the BRIGHT domains topic_ids repeat across domains
            given = {(r["domain"], r["topic_id"]): r["text"] for r in map(json.loads, filter(str.strip, f))}
    ids, texts = [], []
    for c in json.load(open(f"{DATA}/{dom}/benchmark_{split}.json", encoding="utf-8")):
        for t in c["turns"]:
            if not (t.get("gold_doc_ids") or t.get("supporting_doc_ids")):
                continue
            qid = f"{c['id']}_turn_{t['turn_id']}"
            if given is not None:
                text = given[(dom, qid)]
            else:
                text, h = t["query"], t.get("conversation_history", "")
                if qset == "hist" and h and h != "No previous conversation.":
                    text += f"\n\nConversation History:\n{h}"
            ids.append(qid); texts.append(text)
    return ids, texts

def ndcg10(run, dom, split):
    """run: {qid: {doc: score}}. A turn missing from the run scores 0, as on the leaderboard."""
    qrels = {}
    for ln in open(f"{DATA}/{dom}/qrels_{split}.txt"):
        p = ln.split()
        if len(p) == 4:
            qrels.setdefault(p[0], {})[p[2]] = int(p[3])
    res = pytrec_eval.RelevanceEvaluator(qrels, {"ndcg_cut.10"}).evaluate(run)
    return sum(v["ndcg_cut_10"] for v in res.values()) / len(qrels)

def read_run(path):
    run = {}
    for ln in open(path, encoding="utf-8"):
        p = ln.split()
        if len(p) == 6:
            run.setdefault(p[0], {})[p[2]] = float(p[4])
    return run

def write_run(path, run, tag):
    """Writes to a temporary file first, then renames it, so an interrupted write never
    leaves a partial file. Scores are written strictly decreasing and with spare digits:
    pytrec_eval breaks score ties by docno in reverse order, which in five domains would
    systematically rank _neg_ above _doc_. Tied scores keep the dict order (that is,
    the first-stage order)."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path + ".tmp", "w", encoding="utf-8") as f:
        for q, docs in run.items():
            last = None
            for r, (d, sc) in enumerate(sorted(docs.items(), key=lambda x: -x[1]), 1):
                sc = float(sc)
                if last is not None and sc >= last:
                    sc = last - 1e-9 * max(1.0, abs(last))
                last = sc
                f.write(f"{q}\tQ0\t{d}\t{r}\t{sc:.12g}\t{tag}\n")
    os.replace(path + ".tmp", path)

def strata(rs):
    """Slices for the generation tables. All rows ("total"); for 2c, by the qrels label (all/part/none)
    plus "clean" misses (none, and the gold never appeared in earlier turns, so the answer cannot be
    recovered from the history); for 2b and nodoc, the first turn (no history) and the rest."""
    yield "total", rs
    if rs and rs[0]["mode"] == "2c":
        for x in ("all", "part", "none"):
            yield x, [r for r in rs if r["suff"] == x]
        yield "clean", [r for r in rs if r["suff"] == "none" and r.get("gold_in_prior") == 0]
    else:
        yield "T1", [r for r in rs if r.get("turn_id") == 1]
        yield "T2+", [r for r in rs if (r.get("turn_id") or 0) > 1]

# ── log: everything the cells print, timestamped → Drive/reteco/logs ────────
import sys
class _Tee:
    def __init__(self, out, path):
        self.out, self.f, self.bol = out, open(path, "a", encoding="utf-8"), True
    def write(self, x):
        self.out.write(x)
        try:
            for part in x.splitlines(True):
                if self.bol:
                    self.f.write(time.strftime("%m-%d %H:%M:%S "))
                self.f.write(part)
                self.bol = part.endswith("\n")
            self.f.flush()
        except Exception:
            pass
        return len(x)
    def flush(self):
        self.out.flush()
    def __getattr__(self, k):
        return getattr(self.out, k)
os.makedirs(f"{DRIVE}/logs", exist_ok=True)
if not isinstance(getattr(sys.stdout, "_reteco_log", None), str):
    LOG_PATH = f"{DRIVE}/logs/{globals().get('NB', 'notebook')}_{time.strftime('%Y%m%d_%H%M%S')}.log"
    sys.stdout = _Tee(sys.stdout, LOG_PATH)
    sys.stdout._reteco_log = LOG_PATH
LOG_PATH = sys.stdout._reteco_log
print("log:", LOG_PATH)

In [ ]:
%%writefile /content/gen_worker.py
"""Answers of one reader for all jobs (2b, and 2c for each first stage). Run from
generate.ipynb as a separate process per reader, so that vLLM reliably frees GPU
memory before the next model. The summary cell also imports this module."""
import json, math, os, re, sys, time

OCC_STOP = [151643, 151645, 151683]   # <|endoftext|> <|im_end|> <|answer_end|>, from the OCC generation_config
CONTROL_SYSTEM = (
    "You are a helpful assistant in a multi-turn conversation. Answer the user's latest message "
    "using only the numbered documents attached to it.\n"
    "Reply in this format:\n"
    "Line 1: ANSWERABLE if the documents contain enough information to answer, otherwise UNANSWERABLE.\n"
    "Next lines: if ANSWERABLE, the answer in 2-5 sentences in a natural conversational tone; "
    "if UNANSWERABLE, say briefly what information is missing.")


def official_history(h, prev):
    """History limited to exactly what the official conversation_history contains: the
    (question, answer) pairs whose answer appears there; if older turns are compressed into
    "Earlier discussion covered: ...", that summary is prepended to the first question."""
    if not h or h.strip() == "No previous conversation.":
        return []
    hist = [(q, a) for q, a in prev if a and a in h]
    if "Earlier discussion covered" in h and hist:
        hist[0] = (h.split("\n\nRecent conversation:")[0].strip() + "\n\n" + hist[0][0], hist[0][1])
    return hist


def load_turns(data, domains, split):
    """Turns that have gold documents. history is the official dialogue history (official_history).
    reference is the gold answer of the CURRENT turn: used only by the audit and the judge,
    never in the reader prompt. Gold is only what appears in qrels (3 gold ids in politics
    are missing from the corpus). gold_in_prior counts the gold documents of this turn that
    were already gold for earlier turns: then the answer may be in the history."""
    out = []
    for dom in domains:
        qrels = {tuple(l.split()[0:3:2]) for l in open(f"{data}/{dom}/qrels_{split}.txt") if l.strip()}
        for c in json.load(open(f"{data}/{dom}/benchmark_{split}.json", encoding="utf-8")):
            prev, seen = [], set()
            for t in c["turns"]:
                qid = f"{c['id']}_turn_{t['turn_id']}"
                gold = list(dict.fromkeys(d for d in t.get("gold_doc_ids") or t.get("supporting_doc_ids") or []
                                          if (qid, d) in qrels))
                if gold:
                    out.append({"domain": dom, "topic_id": qid, "turn_id": t["turn_id"], "query": t["query"],
                                "history": official_history(t.get("conversation_history", ""), prev),
                                "gold": gold, "gold_in_prior": len(set(gold) & seen),
                                "reference": t.get("answer", "")})
                prev.append((t["query"], t.get("answer", "")))
                seen |= set(gold)
    return out


def read_top(path, k):
    run = {}
    for ln in open(path, encoding="utf-8"):
        p = ln.split()
        if len(p) == 6:
            run.setdefault(p[0], []).append((int(p[3]), p[2]))
    return {q: [d for _, d in sorted(v)[:k]] for q, v in run.items()}


def contexts(C, turns, split, job):
    """"2b" → gold documents; "nodoc" → no documents (what the model knows on its own);
    "2c-<run>" → top-K from Drive/reteco/runs/<run>_<split>."""
    if job == "2b":
        return {(t["domain"], t["topic_id"]): t["gold"] for t in turns}
    if job == "nodoc":
        return {(t["domain"], t["topic_id"]): [] for t in turns}
    if job.startswith("ctrl-"):
        # judge-leniency control: top-K of the same stage for a turn from ANOTHER dialogue in the
        # same domain (half the dialogue list away, same turn number or else the first turn), with
        # this turn's own gold removed, so the context is known not to contain the answer
        tops = {dom: read_top(f"{C['drive']}/runs/{job[5:]}_{split}/{dom}/run.trec", 2 * C["k_ctx"])
                for dom in C["domains"]}
        out, by_dom = {}, {}
        for t in turns:
            by_dom.setdefault(t["domain"], []).append(t)
        for dom, ts in by_dom.items():
            dlg = sorted({t["topic_id"].rsplit("_turn_", 1)[0] for t in ts})
            ids = {t["topic_id"] for t in ts}
            for t in ts:
                d, k = t["topic_id"].rsplit("_turn_", 1)
                other = dlg[(dlg.index(d) + len(dlg) // 2) % len(dlg)]
                src = f"{other}_turn_{k}" if f"{other}_turn_{k}" in ids else f"{other}_turn_1"
                gold = set(t["gold"])
                out[(dom, t["topic_id"])] = [x for x in tops[dom].get(src, []) if x not in gold][:C["k_ctx"]]
        return out
    tops = {dom: read_top(f"{C['drive']}/runs/{job[3:]}_{split}/{dom}/run.trec", C["k_ctx"])
            for dom in C["domains"]}
    return {(t["domain"], t["topic_id"]): tops[t["domain"]].get(t["topic_id"], []) for t in turns}


def sufficiency(gold, ctx):
    """Number of gold documents in the context, and the label: all = all gold that fits in K; none = no gold."""
    n = len(set(gold) & set(ctx))
    full = min(len(gold), len(ctx)) if ctx else len(gold)
    return n, ("none" if n == 0 else "all" if n >= full else "part")


def corpus_texts(data, need):
    """need: {domain: {doc_id}} → {(domain, doc_id): text}."""
    out = {}
    for dom, ids in need.items():
        for ln in open(f"{data}/{dom}/documents.jsonl", encoding="utf-8"):
            if ln.strip():
                d = json.loads(ln)
                if d["doc_id"] in ids:
                    out[(dom, d["doc_id"])] = d["content"]
    return out


def history_msgs(t):
    m = []
    for q, a in t["history"]:
        m += [{"role": "user", "content": q}, {"role": "assistant", "content": a}]
    return m


class OCC:
    """The OCC chat template itself places the documents in the last user turn; generation
    starts with <|query_analysis_start|>, the status is <|status_start|>(UN)ANSWERABLE<|status_end|>."""
    stop, skip_special, marker = OCC_STOP, False, "<|status_start|>"
    STATUS = re.compile(r"<\|status_start\|>(.*?)(?:<\|status_end\|>|$)", re.S)
    ANSWER = re.compile(r"<\|answer_start\|>(.*?)(?:<\|answer_end\|>|<\|im_end\|>|$)", re.S)
    SPECIAL = re.compile(r"<\|[a-z_]+\|>")

    def __init__(self, tok):
        self.tok = tok

    def prompt(self, t, docs):
        msgs = history_msgs(t) + [{"role": "user", "content": t["query"]}]
        return self.tok.apply_chat_template(msgs, documents=[{"text": d} for d in docs],
                                            add_generation_prompt=True, tokenize=False,
                                            enable_thinking=False)

    def parse(self, raw):
        m = self.STATUS.search(raw)
        status = self.SPECIAL.sub("", m.group(1)).strip().upper() if m else ""
        m = self.ANSWER.search(raw)
        return status, (self.SPECIAL.sub("", m.group(1)).strip() if m else "")

    def prefix(self, raw):
        """Output up to the verdict position: the marker plus the newline after it (OCC always
        writes "<|status_start|>\nANSWERABLE"), so that the next token is the verdict."""
        i = raw.find(self.marker)
        if i < 0:
            return None
        j = i + len(self.marker)
        while j < len(raw) and raw[j] in "\r\n":
            j += 1
        return raw[:j]


class Control:
    """Qwen3 readers (the control prompt): same input, status on the first line."""
    stop, skip_special = None, True

    def __init__(self, tok):
        self.tok = tok

    def prompt(self, t, docs):
        numbered = "\n\n".join(f"[{i}] {d}" for i, d in enumerate(docs, 1)) or "(none)"
        msgs = ([{"role": "system", "content": CONTROL_SYSTEM}] + history_msgs(t)
                + [{"role": "user", "content": f"Documents:\n{numbered}\n\nQuestion: {t['query']}"}])
        return self.tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False,
                                            enable_thinking=False)

    def parse(self, raw):
        first, _, rest = raw.strip().partition("\n")
        m = re.search(r"\b(UN)?ANSWERABLE\b", first.upper())
        if not m:
            return "", raw.strip()
        return m.group(0), rest.strip()

    def prefix(self, raw):
        """Everything before the status word on the first line (the model may start with "Line 1:" or **)."""
        lead = len(raw) - len(raw.lstrip())
        m = re.search(r"\b(?:UN)?ANSWERABLE\b", raw[lead:].split("\n", 1)[0], re.I)
        return raw[:lead + m.start()] if m else None


def p_status(tok, lp):
    """Probability of ANSWERABLE at the status position: summed probabilities of tokens that are
    prefixes of "ANSWERABLE" versus "UNANSWERABLE" among the top 20 (ANS... vs UN...). mass is
    how much probability fell on these two options at all."""
    a = u = 0.0
    for tid, x in lp.items():
        s = tok.decode([tid]).strip()
        if s and "UNANSWERABLE".startswith(s):
            u += math.exp(x.logprob)
        elif s and "ANSWERABLE".startswith(s):
            a += math.exp(x.logprob)
    return (round(a / (a + u), 6) if a + u > 0 else None), round(a + u, 6)


def main(reader_id, cfg_path):
    from vllm import LLM, SamplingParams
    C = json.load(open(cfg_path))
    short = reader_id.split("/")[-1]
    todo_any = False
    for split in C["splits"]:
        keys = {(t["domain"], t["topic_id"]) for t in load_turns(C["data"], C["domains"], split)}
        for job in C["jobs"]:
            path = f"{C['drive']}/gen/{short}_{job}_{split}.jsonl"
            done = ({(r["domain"], r["topic_id"]) for r in map(json.loads, open(path, encoding="utf-8"))}
                    if os.path.exists(path) else set())
            todo_any |= bool(keys - done)
    if not todo_any:
        print(f"{short}: everything is already done, skipping model load", flush=True)
        return
    llm = LLM(model=reader_id, dtype=C["dtype"], max_model_len=C["maxlen"],
              gpu_memory_utilization=0.88, seed=0)
    tok = llm.get_tokenizer()
    R = (OCC if "OCC" in reader_id else Control)(tok)
    # OCC: greedy only, no repetition_penalty, otherwise the structured format breaks
    sp = SamplingParams(temperature=0.0, max_tokens=C["max_new"], stop_token_ids=R.stop,
                        skip_special_tokens=R.skip_special, spaces_between_special_tokens=False)
    sp1 = SamplingParams(temperature=0.0, max_tokens=1, logprobs=20)
    short = reader_id.split("/")[-1]
    limit = C["maxlen"] - C["max_new"]
    ntok = lambda p: len(tok(p, add_special_tokens=False).input_ids)
    os.makedirs(f"{C['drive']}/gen", exist_ok=True)
    for split in C["splits"]:
        turns = load_turns(C["data"], C["domains"], split)
        jobs = [(j, contexts(C, turns, split, j)) for j in C["jobs"]]
        need = {}
        for _, ctx in jobs:
            for (dom, _), ids in ctx.items():
                need.setdefault(dom, set()).update(ids)
        text = corpus_texts(C["data"], need)
        for job, ctx in jobs:
            path = f"{C['drive']}/gen/{short}_{job}_{split}.jsonl"
            done = set()
            if os.path.exists(path):
                done = {(r["domain"], r["topic_id"]) for r in map(json.loads, open(path, encoding="utf-8"))}
            prompts, keep, skipped = [], [], 0
            for t in turns:
                if (t["domain"], t["topic_id"]) in done:
                    continue
                ids = ctx[(t["domain"], t["topic_id"])]
                docs = [text[(t["domain"], d)] for d in ids]
                p, trimmed = R.prompt(t, docs), False
                if ntok(p) > limit:      # does not fit: trim every document rather than drop gold
                    p, trimmed = R.prompt(t, [" ".join(d.split()[:C["trim_words"]]) for d in docs]), True
                if ntok(p) > limit:
                    skipped += 1
                    continue
                prompts.append(p)
                keep.append((t, ids, trimmed))
            print(f"{short} · {job} · {split}: to do {len(prompts)}, done {len(done)}, "
                  f"too long {skipped}", flush=True)
            t0 = time.time()
            with open(path, "a", encoding="utf-8") as f:
                for s in range(0, len(prompts), C["chunk"]):
                    outs = llm.generate(prompts[s:s + C["chunk"]], sp, use_tqdm=False)
                    raws = [o.outputs[0].text for o in outs]
                    # second pass: prompt + answer up to the status position, one token, top-20 logprobs
                    pref = [(i, R.prefix(r)) for i, r in enumerate(raws)]
                    pref = [(i, x) for i, x in pref if x is not None]
                    probs = {}
                    if pref:
                        o1 = llm.generate([prompts[s + i] + x for i, x in pref], sp1, use_tqdm=False)
                        for (i, _), o in zip(pref, o1):
                            probs[i] = p_status(tok, o.outputs[0].logprobs[0])
                    for i, (o, raw) in enumerate(zip(outs, raws)):
                        t, ids, trimmed = keep[s + i]
                        status, answer = R.parse(raw)
                        n_in, suff = sufficiency(t["gold"], ids)
                        p, mass = probs.get(i, (None, 0.0))
                        f.write(json.dumps({
                            "domain": t["domain"], "topic_id": t["topic_id"], "turn_id": t["turn_id"],
                            "split": split, "reader": reader_id, "mode": job.split("-")[0],
                            "first_stage": job[3:] if job.startswith("2c-") else None,
                            "context_ids": ids, "n_gold": len(t["gold"]), "gold_in_prior": t["gold_in_prior"],
                            "gold_in_ctx": n_in, "suff": suff, "status": status, "answer": answer,
                            "p_answerable": p, "p_mass": mass, "finish": o.outputs[0].finish_reason,
                            "trimmed": trimmed, "raw": raw}, ensure_ascii=False) + "\n")
                    f.flush()
                    k, el = s + len(outs), time.time() - t0
                    print(f"  {k}/{len(prompts)} · {el / 60:.1f} min · ≈{el / k * (len(prompts) - k) / 60:.0f} min left",
                          flush=True)


if __name__ == "__main__":
    main(sys.argv[1], sys.argv[2])

In [ ]:
%%writefile /content/art_worker.py
"""OCC answers on synthetic unanswerability (SQuAD 2.0, MuSiQue), with the same prompt
and decoding as gen_worker. Runs as a separate process so that vLLM releases the GPU."""
import json, os, sys
import gen_worker as G


def main(reader_id, cfg_path):
    from vllm import LLM, SamplingParams
    C = json.load(open(cfg_path))
    items = [json.loads(l) for l in open(C["items"], encoding="utf-8")]
    done = set()
    if os.path.exists(C["out"]):
        done = {json.loads(l)["id"] for l in open(C["out"], encoding="utf-8")}
    todo = [it for it in items if it["id"] not in done]
    print(f"synthetic: {len(todo)} to do, {len(done)} done", flush=True)
    if not todo:
        return
    llm = LLM(model=reader_id, dtype=C["dtype"], max_model_len=C["maxlen"], gpu_memory_utilization=0.88, seed=0)
    tok = llm.get_tokenizer()
    R = G.OCC(tok)
    sp = SamplingParams(temperature=0.0, max_tokens=C["max_new"], stop_token_ids=R.stop,
                        skip_special_tokens=False, spaces_between_special_tokens=False)
    limit = C["maxlen"] - C["max_new"]
    keep, prompts = [], []
    for it in todo:
        p = R.prompt({"history": [], "query": it["query"]}, it["docs"])
        if len(tok(p, add_special_tokens=False).input_ids) <= limit:
            keep.append(it); prompts.append(p)
    with open(C["out"], "a", encoding="utf-8") as f:
        for s in range(0, len(prompts), C["chunk"]):
            outs = llm.generate(prompts[s:s + C["chunk"]], sp, use_tqdm=False)
            for it, o in zip(keep[s:s + C["chunk"]], outs):
                f.write(json.dumps({"id": it["id"], "raw": o.outputs[0].text}, ensure_ascii=False) + "\n")
            f.flush()
            print(f"  {s + len(outs)}/{len(prompts)}", flush=True)


if __name__ == "__main__":
    main(sys.argv[1], sys.argv[2])

In [ ]:
# ── 7. Synthetic unanswerability: SQuAD 2.0 and MuSiQue, balanced classes ───
import random
import pandas as pd
sys.path.insert(0, "/content")
import gen_worker as G
TR = f"{DRIVE}/transfer"
os.makedirs(TR, exist_ok=True)
PQ = "https://huggingface.co/api/datasets/{}/parquet/{}/train/0.parquet"

def truthy(v):
    return v.strip().lower() == "true" if isinstance(v, str) else bool(v)

def sample(df, is_ans, make, rng):
    """N_TRAIN + N_VAL examples per class; rows are built only for the sampled ones."""
    out = []
    for cls in (True, False):
        idx = [i for i, v in enumerate(is_ans) if v == cls]
        rng.shuffle(idx)
        for k, i in enumerate(idx[:N_TRAIN + N_VAL]):
            x = make(df.iloc[i])
            x["id"] += "_a" if cls else "_u"     # in MuSiQue an answerable/unanswerable pair shares one id
            out.append({**x, "answerable": cls, "part": "train" if k < N_TRAIN else "val"})
    return out

ITEMS = f"{TR}/art_items.jsonl"
if not os.path.exists(ITEMS):
    rng = random.Random(SEED)
    sq = pd.read_parquet(PQ.format("rajpurkar/squad_v2", "squad_v2"))
    mq = pd.read_parquet(PQ.format("bdsaglam/musique", "default"))
    items = (sample(sq, [len(a["text"]) > 0 for a in sq["answers"]],
                    lambda r: {"id": f"squad_{r['id']}", "source": "squad", "query": r["question"],
                               "docs": [r["context"]]}, rng)
             + sample(mq, [truthy(v) for v in mq["answerable"]],
                      lambda r: {"id": f"musique_{r['id']}", "source": "musique", "query": r["question"],
                                 "docs": [str(p.get("paragraph_text") or "") for p in r["paragraphs"]]}, rng))
    with open(ITEMS + ".tmp", "w", encoding="utf-8") as f:
        for x in items:
            f.write(json.dumps(x, ensure_ascii=False) + "\n")
    os.replace(ITEMS + ".tmp", ITEMS)
ART = {x["id"]: x for x in map(json.loads, open(ITEMS, encoding="utf-8"))}
print("synthetic examples:", len(ART), "· answerable:", sum(x["answerable"] for x in ART.values()))

In [ ]:
# ── 8. OCC responses: synthetic + RETECO train/dev (existing ones in gen/ are skipped) ──
def stream(cmd):
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env={**os.environ, "VLLM_LOGGING_LEVEL": "WARNING"})
    for line in p.stdout:
        print(line, end="")
    return p.wait()

def generate_all():
    base = {"drive": DRIVE, "data": DATA, "domains": DOMAINS, "k_ctx": K_CTX, "max_new": MAX_NEW, "maxlen": MAXLEN,
            "trim_words": TRIM_WORDS, "chunk": CHUNK, "dtype": DTYPE}
    for split, stages in (("dev", DEV_STAGES), ("train", TRAIN_STAGES)):
        for fs in stages:
            assert os.path.isdir(f"{RUNS}/{fs}_{split}"), f"missing run {RUNS}/{fs}_{split}"
        json.dump({**base, "splits": [split], "jobs": [f"2c-{fs}" for fs in stages]}, open(f"/content/cfg_{split}.json", "w"))
        assert stream([sys.executable, "/content/gen_worker.py", READER, f"/content/cfg_{split}.json"]) == 0
    json.dump({**base, "items": ITEMS, "out": f"{TR}/art_gen.jsonl"}, open("/content/cfg_art.json", "w"))
    assert stream([sys.executable, "/content/art_worker.py", READER, "/content/cfg_art.json"]) == 0

if EVAL_ONLY:
    print("EVAL_ONLY: generation skipped, features come from existing", f"{TR}/feat_*.npz")
else:
    if "lm" in globals():      # rerun in the same session: the HF model from cell 9 holds the GPU
        del lm
        import gc; gc.collect(); torch.cuda.empty_cache()
    generate_all()

In [ ]:
# ── 9. Features: one model pass over the prompt + trace up to <|status_start|> ──
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from packaging.version import Version
DK = "dtype" if Version(transformers.__version__) >= Version("4.56") else "torch_dtype"
tok = AutoTokenizer.from_pretrained(READER)      # EVAL_ONLY: tokenizer only, to match rows against gen/
R = G.OCC(tok)
if not EVAL_ONLY:
    lm = AutoModelForCausalLM.from_pretrained(READER, **{DK: torch.bfloat16 if DTYPE == "bfloat16" else torch.float16})
    lm = lm.cuda().eval()
    ANS_IDS, UN_IDS = [], []
    for i in range(len(tok)):        # tokens that are verdict prefixes: ANS… vs UN…
        s = tok.decode([i]).strip()
        if s and "UNANSWERABLE".startswith(s):
            UN_IDS.append(i)
        elif s and "ANSWERABLE".startswith(s):
            ANS_IDS.append(i)

@torch.inference_mode()
def features(prompt, raw):
    """→ (hidden states of the LAYERS at the status position, p_answerable, trace length,
    mean trace log-probability), or None if OCC did not reach the status."""
    pre = R.prefix(raw)            # trace up to the verdict itself (marker + newline)
    if pre is None:
        return None
    p_ids = tok(prompt, add_special_tokens=False).input_ids
    t_ids = tok(pre, add_special_tokens=False).input_ids
    if len(p_ids) + len(t_ids) > MAXLEN:
        return None
    ids = torch.tensor([p_ids + t_ids], device="cuda")
    try:
        out = lm(input_ids=ids, output_hidden_states=True, logits_to_keep=len(t_ids) + 1)
    except TypeError:
        out = lm(input_ids=ids, output_hidden_states=True)
    logits = out.logits[0, -(len(t_ids) + 1):].float()
    lp = logits[:-1].log_softmax(-1).gather(1, ids[0, len(p_ids):, None]).squeeze(1)
    pr = logits[-1].softmax(-1)
    a, u = pr[ANS_IDS].sum().item(), pr[UN_IDS].sum().item()
    h = torch.stack([out.hidden_states[l][0, -1] for l in LAYERS]).half().cpu().numpy()
    return h, (a / (a + u) if a + u > 0 else 0.5), len(t_ids), lp.mean().item()

def build(name, rows_fn, every=500):
    """rows_fn() → [(prompt, raw response, label 1/0/-1, meta, group, key)] → Drive/reteco/transfer/feat_<name>.npz.
    An existing file is simply loaded (rows_fn is not called); with EVAL_ONLY and no file, returns None.
    Every `every` rows a chunk is saved to Drive; after an interruption, work resumes from the next chunk."""
    path = f"{TR}/feat_{name}.npz"
    if os.path.exists(path):
        return dict(np.load(path, allow_pickle=True))
    if EVAL_ONLY:
        print(f"missing {path}: EVAL_ONLY, skipping")
        return None
    rows = rows_fn()
    parts = sorted(glob.glob(f"{TR}/feat_{name}.part*.npz"))
    start = sum(int(np.load(p)["n_rows"]) for p in parts)     # rows already processed
    t0, buf, chunk0 = time.time(), [], start
    for k in tqdm(range(start, len(rows)), desc=name):
        prompt, raw, y, meta, grp, key = rows[k]
        f = features(prompt, raw)
        if f is not None:
            buf.append((f, int(R.parse(raw)[0] == "ANSWERABLE"), y, meta, grp, key))
        if k + 1 - chunk0 == every or k == len(rows) - 1:
            d = {"n_rows": np.array(k + 1 - chunk0)}
            chunk0 = k + 1
            if buf:
                d.update(H=np.stack([b[0][0] for b in buf]), p=np.array([b[0][1] for b in buf]),
                         tlen=np.array([b[0][2] for b in buf]), tlp=np.array([b[0][3] for b in buf]),
                         status=np.array([b[1] for b in buf]), y=np.array([b[2] for b in buf]),
                         meta=np.array([b[3] for b in buf]), grp=np.array([b[4] for b in buf]),
                         key=np.array([b[5] for b in buf]))
            pp = f"{TR}/feat_{name}.part{len(parts):04d}.npz"
            np.savez(pp + ".tmp.npz", **d)
            os.replace(pp + ".tmp.npz", pp)
            parts.append(pp); buf = []
    loaded = [dict(np.load(p, allow_pickle=True)) for p in sorted(set(parts))]
    loaded = [x for x in loaded if "y" in x]
    keys = [k for k in ["H", "p", "tlen", "tlp", "status", "y", "meta", "grp", "key"]
            if all(k in x for x in loaded)]          # chunks written before key was added lack it
    d = {k: np.concatenate([x[k] for x in loaded]) for k in keys}
    np.savez(path + ".tmp.npz", **d)
    os.replace(path + ".tmp.npz", path)
    for p in set(parts):
        os.remove(p)
    print(f"{name}: {len(d['y'])} examples out of {len(rows)} (the rest have no status), {(time.time() - t0) / 60:.1f} min")
    return d

def art_rows(part):
    art_raw = {r["id"]: r["raw"] for r in map(json.loads, open(f"{TR}/art_gen.jsonl", encoding="utf-8"))}
    return [(R.prompt({"history": [], "query": x["query"]}, x["docs"]), art_raw[i], int(x["answerable"]), x["source"], i, i)
            for i, x in ART.items() if x["part"] == part and i in art_raw]

gen_path = lambda split, fs: f"{DRIVE}/gen/{READER.split('/')[-1]}_2c-{fs}_{split}.jsonl"

def reteco_rows(split, stages):
    turns = {(t["domain"], t["topic_id"]): t for t in G.load_turns(DATA, DOMAINS, split)}
    recs = [r for fs in stages for r in map(json.loads, open(gen_path(split, fs), encoding="utf-8"))]
    need = {}
    for r in recs:
        need.setdefault(r["domain"], set()).update(r["context_ids"])
    text = G.corpus_texts(DATA, need)
    rows = []
    for r in recs:
        docs = [text[(r["domain"], d)] for d in r["context_ids"]]
        if r["trimmed"]:
            docs = [" ".join(d.split()[:TRIM_WORDS]) for d in docs]
        y = {"all": 1, "none": 0}.get(r["suff"], -1)
        grp = f"{r['domain']}|{r['topic_id'].rsplit('_turn_', 1)[0]}"     # dialogue: its turns are never split up
        rows.append((R.prompt(turns[(r["domain"], r["topic_id"])], docs), r["raw"], y, r["first_stage"], grp,
                     f"{r['domain']}|{r['topic_id']}"))
    return rows

def with_keys(d, fs, src, split="dev"):
    """Key "domain|topic_id" of every row, used to join with the labels in suff/. Newer files
    store it themselves (key); older files lack it. build walked the records of the stage's
    gen file in file order and dropped those where OCC did not reach the status (R.prefix is
    None) or where the prompt plus trace exceeded MAXLEN. Recovery repeats that pass: records
    without a status are skipped, and each feature row is matched to a record by (dialogue,
    qrels label, verdict, trace length in tokens); on a mismatch the record was dropped for
    length and the next one is tried. If matching fails before the end (the gen file changed
    after build), an error is raised."""
    if "key" in d:
        return d
    recs = [r for r in map(json.loads, open(gen_path(split, fs), encoding="utf-8")) if R.prefix(r["raw"]) is not None]
    sig = [(f"{r['domain']}|{r['topic_id'].rsplit('_turn_', 1)[0]}", {"all": 1, "none": 0}.get(r["suff"], -1),
            int(R.parse(r["raw"])[0] == "ANSWERABLE"), len(tok(R.prefix(r["raw"]), add_special_tokens=False).input_ids))
           for r in recs]
    rows = list(zip(map(str, d["grp"]), map(int, d["y"]), map(int, d["status"]), map(int, d["tlen"])))
    for n in (4, 3):               # 3: without the trace length, in case the tokenizer counted it differently
        keys, j = [], 0
        for row in rows:
            while j < len(sig) and sig[j][:n] != row[:n]:
                j += 1
            if j == len(sig):
                break
            keys.append(f"{recs[j]['domain']}|{recs[j]['topic_id']}"); j += 1
        if len(keys) == len(rows):
            break
    assert len(keys) == len(rows), (f"features {fs} do not match {gen_path(split, fs)}: the file changed after build; "
                                    f"delete {src} and recompute (EVAL_ONLY = False)")
    print(f"  {fs}: keys recovered from the gen-file order ({len(rows)} rows; matched on {n} fields; "
          f"dropped for length: {len(recs) - len(rows)})")
    return {**d, "key": np.array(keys)}

F = {}
for part in ("train", "val"):
    F[f"art_{part}"] = build(f"art_{part}", lambda part=part: art_rows(part))
F["ret_train"] = build("ret_train", lambda: reteco_rows("train", TRAIN_STAGES))
_miss = [k for k, v in F.items() if v is None]
assert not _miss, f"missing features {_miss} in {TR}: run the full pass first (EVAL_ONLY = False, GPU)"

# RETECO dev: one file per first stage (feat_ret_dev-<stage>.npz), so a new stage leaves the existing
# ones untouched. A combined feat_ret_dev.npz from earlier runs (three BM25 stages) is read and split by stage.
OLD = f"{TR}/feat_ret_dev.npz"
old = dict(np.load(OLD, allow_pickle=True)) if os.path.exists(OLD) else None
DEV, DEV_OK = [], []
for fs in DEV_STAGES:
    if old is not None and fs in set(old["meta"].tolist()):
        d, src = {k: v[old["meta"] == fs] for k, v in old.items()}, OLD
    else:
        d, src = build(f"ret_dev-{fs}", lambda fs=fs: reteco_rows("dev", [fs])), f"{TR}/feat_ret_dev-{fs}.npz"
    if d is not None and len(d["y"]):
        DEV.append(with_keys(d, fs, src)); DEV_OK.append(fs)
assert DEV, f"no RETECO dev features in {TR}"
F["ret_dev"] = {k: np.concatenate([d[k] for d in DEV]) for k in set.intersection(*(set(d) for d in DEV))}
print("RETECO dev, stages with features:", DEV_OK, "· rows:", len(F["ret_dev"]["y"]))

In [ ]:
# ── 10. Detectors: train on synthetic data or RETECO train, evaluate on RETECO dev ──
import csv
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, balanced_accuracy_score

def lab(d):
    m = d["y"] >= 0                    # the "part" label (only some of the gold) is excluded
    return {k: v[m] for k, v in d.items()}

A_tr, A_va, R_tr, R_dev = (lab(F[k]) for k in ("art_train", "art_val", "ret_train", "ret_dev"))
rng = np.random.default_rng(SEED)
from sklearn.model_selection import GroupShuffleSplit
# 20% of RETECO train dialogues select the layer for the in-domain setting; split by dialogue, not by row:
# each turn appears once per first stage, and the turns of a dialogue share its history
_, _hi = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED).split(R_tr["y"], groups=R_tr["grp"]))
hold = np.zeros(len(R_tr["y"]), bool); hold[_hi] = True
sub = lambda d, m: {k: v[m] for k, v in d.items()}
R_fit, R_hold = sub(R_tr, ~hold), sub(R_tr, hold)
auc = lambda y, s: round(float(roc_auc_score(y, s)), 4) if len(set(y)) == 2 else float("nan")
trace = lambda d: np.stack([d["tlen"], d["tlp"]], 1)
fit = lambda X, y: make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=2000)).fit(X, y)

def by_stage(score_fn):
    return {fs: auc(R_dev["y"][R_dev["meta"] == fs], score_fn(sub(R_dev, R_dev["meta"] == fs))) for fs in DEV_OK}

rows, DET = [], []                     # DET: the trained detectors; cell 11 evaluates the same ones
def row(name, src, layer, s_va, s_dev_fn):
    DET.append((name, src, layer, s_dev_fn))
    rows.append({"detector": name, "train": src, "layer": layer, "auroc_val": s_va,
                 "auroc_dev": auc(R_dev["y"], s_dev_fn(R_dev)), **by_stage(s_dev_fn)})

# Detector and training-source names are data values in transfer.csv and transfer_content.csv;
# baselines_v2.py matches them in transfer_content.csv (XNAME). synthetic = SQuAD 2.0 + MuSiQue.
row("OCC verdict", "—", "", auc(A_va["y"], A_va["status"]), lambda d: d["status"])
row("p_status", "—", "", auc(A_va["y"], A_va["p"]), lambda d: d["p"])
for src, tr, va in (("synthetic", A_tr, A_va), ("RETECO train", R_fit, R_hold)):
    m = fit(trace(tr), tr["y"])
    row("trace", src, "", auc(va["y"], m.predict_proba(trace(va))[:, 1]), lambda d, m=m: m.predict_proba(trace(d))[:, 1])
    best = max(range(len(LAYERS)), key=lambda j: auc(va["y"], fit(tr["H"][:, j], tr["y"]).predict_proba(va["H"][:, j])[:, 1]))
    m = fit(tr["H"][:, best], tr["y"])
    row("probe", src, LAYERS[best], auc(va["y"], m.predict_proba(va["H"][:, best])[:, 1]),
        lambda d, m=m, j=best: m.predict_proba(d["H"][:, j])[:, 1])

print(f"RETECO dev: {int(R_dev['y'].sum())} with gold / {int((1 - R_dev['y']).sum())} without; "
      f"synthetic val: {len(A_va['y'])}")
print("OCC verdict on dev, balanced accuracy:",
      round(balanced_accuracy_score(R_dev["y"], R_dev["status"]), 4))
cols = ["auroc_val", "auroc_dev"] + DEV_OK
short = lambda fs: fs.replace("dense_hanhainebula_reason-embed-qwen3-4b-0928", "RE")    # hyb_RE_q2d
print(f"{'detector':<14}{'trained on':<16}{'layer':>5}" + "".join(f"{short(c)[-11:]:>12}" for c in cols))
for r in rows:
    print(f"{r['detector']:<14}{r['train']:<16}{str(r['layer']):>5}" + "".join(f"{r[c]:>12.4f}" for c in cols))
os.makedirs(f"{DRIVE}/analysis", exist_ok=True)
with open(f"{DRIVE}/analysis/transfer.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
print("\nauroc_val: on the held-out part of the training data; auroc_dev: transfer to RETECO dev 2c"
      "\n(all stages with features pooled), followed by per-stage values.")

In [ ]:
# ── 11. The same detectors under the content label (suff.ipynb) ─────────────
# Label names are data values in transfer_content.csv (baselines_v2.py matches qrels and content, XLAB):
# qrels   — the qrels label: all gold in the top-5 = 1, none = 0;
# content — the content label, the sufficiency judge read the documents: yes = 1, no = 0 (partial and
#           unparsed labels are excluded; rows labelled part by qrels are included);
# clean   — the clean label: yes = 1 vs. no where the answer is also absent from the dialogue history
#           (answered_in_history = false).
# Detectors are not retrained: these are DET from cell 10. Stage "total" = all stages with content labels pooled.
D = F["ret_dev"]                       # all dev rows, including those labelled part by qrels
LAB = {"qrels": D["y"].astype(int), "content": np.full(len(D["y"]), -1), "clean": np.full(len(D["y"]), -1)}
HAVE = []
for fs in DEV_OK:
    path = f"{DRIVE}/suff/{SUFF_JUDGE}_2c-{fs}_dev.jsonl"
    if not os.path.exists(path):
        print("missing labels", path); continue
    S = {f"{r['domain']}|{r['topic_id']}": r for r in map(json.loads, open(path, encoding="utf-8"))}
    idx, miss = np.where(D["meta"] == fs)[0], 0
    for i in idx:
        s = S.get(str(D["key"][i]))
        if s is None:
            miss += 1; continue
        LAB["content"][i] = {"yes": 1, "no": 0}.get(s["label"], -1)
        LAB["clean"][i] = 1 if s["label"] == "yes" else 0 if s["label"] == "no" and s["answered_in_history"] is False else -1
    HAVE.append(fs)
    print(f"{short(fs)}: feature rows {len(idx)}, without a suff label {miss}")
if not HAVE:
    print(f"missing {DRIVE}/suff/{SUFF_JUDGE}_2c-*_dev.jsonl: run suff.ipynb first; evaluation skipped")
else:
    parts = [(fs, D["meta"] == fs) for fs in HAVE] + [("total", np.isin(D["meta"], HAVE))]
    print("\nn (1 / 0):")
    for ln, y in LAB.items():
        print(f"  {ln:<7}" + "  ".join(f"{short(p)} {int((y[m] == 1).sum())}/{int((y[m] == 0).sum())}" for p, m in parts))
    out = []
    print(f"\n{'detector':<14}{'trained on':<16}{'layer':>5}{'label':>8}" + "".join(f"{short(p)[-11:]:>12}" for p, _ in parts))
    for name, src, layer, fn in DET:
        s = fn(D)
        for ln, y in LAB.items():
            vals = []
            for p, m in parts:
                k = m & (y >= 0)
                vals.append(auc(y[k], s[k]))
                out.append({"detector": name, "train": src, "layer": layer, "label": ln, "stage": p,
                            "n_pos": int((y[k] == 1).sum()), "n_neg": int((y[k] == 0).sum()), "auroc": vals[-1]})
            print(f"{name:<14}{src:<16}{str(layer):>5}{ln:>8}" + "".join(f"{v:>12.4f}" for v in vals))
    with open(f"{DRIVE}/analysis/transfer_content.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(out[0])); w.writeheader(); w.writerows(out)
    print(f"\nAUROC of the same detectors; per-stage qrels values match the table above. → {DRIVE}/analysis/transfer_content.csv")

In [ ]:
# ── End. Everything is on Drive; the runtime disconnects so that no compute units are spent ──
print("done:", time.strftime("%Y-%m-%d %H:%M:%S"), "· log:", globals().get("LOG_PATH", "—"))
if AUTO_STOP:
    stop_runtime("notebook finished")